# NYC Taxi Data Analysis with Spark SQL

This notebook loads Yellow and Green taxi data for all months of 2020 and demonstrates SQL querying in Spark.

In [5]:
# Set JAVA_HOME and HADOOP_HOME for Spark
import os
os.environ['JAVA_HOME'] = r'C:\tools\jdk-17.0.18+8'
os.environ['HADOOP_HOME'] = r'C:\tools\hadoop'
# Add both Java and Hadoop bin directories to PATH
os.environ['PATH'] = r'C:\tools\hadoop\bin;' + os.environ['JAVA_HOME'] + r'\bin;' + os.environ.get('PATH', '')

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F

print(f"JAVA_HOME: {os.environ['JAVA_HOME']}")
print(f"HADOOP_HOME: {os.environ['HADOOP_HOME']}")
print(f"PySpark version: {pyspark.__version__}")

JAVA_HOME: C:\tools\jdk-17.0.18+8
HADOOP_HOME: C:\tools\hadoop
PySpark version: 4.1.1


In [6]:
# Create Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('taxi_sql_analysis') \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [7]:
# Green Taxi Schema
green_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("ehail_fee", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("trip_type", types.IntegerType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

# Yellow Taxi Schema
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

print("Schemas defined successfully")

Schemas defined successfully


## Load Yellow Taxi Data

In [8]:
# Load ALL Yellow taxi data for 2020 (all 12 months)
print("Loading all Yellow taxi data for 2020...")
df_yellow = spark.read \
    .option("header", "true") \
    .schema(yellow_schema) \
    .csv("data/yellow/*.csv.gz")

print(f"\nYellow taxi 2020 - Total row count: {df_yellow.count():,}")

Loading all Yellow taxi data for 2020...

Yellow taxi 2020 - Total row count: 24,648,499


## Load Green Taxi Data

In [9]:
# Load ALL Green taxi data for 2020 (all 12 months)
print("Loading all Green taxi data for 2020...")
df_green = spark.read \
    .option("header", "true") \
    .schema(green_schema) \
    .csv("data/green/*.csv.gz")

print(f"\nGreen taxi 2020 - Total row count: {df_green.count():,}")

Loading all Green taxi data for 2020...

Green taxi 2020 - Total row count: 1,734,051


## Prepare Data for Analysis

Standardize column names and add taxi type identifier.

In [10]:
# Rename pickup/dropoff columns to common names and add service type
df_yellow_common = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumn('service_type', F.lit('yellow'))

df_green_common = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumn('service_type', F.lit('green'))

print("Standardized column names")
print(f"Yellow columns: {df_yellow_common.columns}")
print(f"\nGreen columns: {df_green_common.columns}")

Standardized column names
Yellow columns: ['VendorID', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'service_type']

Green columns: ['VendorID', 'pickup_datetime', 'dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge', 'service_type']


In [11]:
# Select common columns for union
common_columns = [
    'VendorID',
    'pickup_datetime',
    'dropoff_datetime',
    'store_and_fwd_flag',
    'RatecodeID',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'fare_amount',
    'extra',
    'mta_tax',
    'tip_amount',
    'tolls_amount',
    'improvement_surcharge',
    'total_amount',
    'payment_type',
    'congestion_surcharge',
    'service_type'
]

df_yellow_selected = df_yellow_common.select(common_columns)
df_green_selected = df_green_common.select(common_columns)

print(f"Selected {len(common_columns)} common columns")

Selected 19 common columns


In [14]:
# Combine both taxi types
df_all_trips = df_yellow_selected.union(df_green_selected)

print(f"Combined dataset - Total trips: {df_all_trips.count():,}")

Combined dataset - Total trips: 26,382,550


## Register as SQL Tables

In [15]:
# Register DataFrames as temporary SQL views
df_yellow_selected.createOrReplaceTempView('yellow_taxi')
df_green_selected.createOrReplaceTempView('green_taxi')
df_all_trips.createOrReplaceTempView('all_trips')

print("Created SQL views:")
print("  - yellow_taxi")
print("  - green_taxi")
print("  - all_trips")

Created SQL views:
  - yellow_taxi
  - green_taxi
  - all_trips


## SQL Queries

Now we can query the data using SQL!

In [ ]:
# Query 1: Total trips by service type
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green_taxi
WHERE
    service_type = 'green'
    AND pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
ORDER BY 2 ASC, 1 ASC
""")

df_green_revenue.show()

{"ts": "2026-03-11 12:50:19.838", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `green` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o28.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `green` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXI

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `green` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 9 pos 4;
'Aggregate [unresolvedordinal(1), unresolvedordinal(2)], ['date_trunc(hour, 'lpep_pickup_datetime) AS hour#115, 'PULocationID AS zone#116, 'SUM('total_amount) AS amount#117, count(1) AS number_records#118L]
+- 'Filter (('service_type = green) AND ('lpep_pickup_datetime >= 2020-01-01 00:00:00))
   +- 'UnresolvedRelation [green], [], false


In [ ]:
# Query 2: Monthly trip trends
result = spark.sql("""
    SELECT 
        YEAR(pickup_datetime) as year,
        MONTH(pickup_datetime) as month,
        service_type,
        COUNT(*) as trips
    FROM all_trips
    GROUP BY YEAR(pickup_datetime), MONTH(pickup_datetime), service_type
    ORDER BY year, month, service_type
""")

result.show(24)

In [ ]:
# Query 3: Top 10 pickup locations (by trip count)
result = spark.sql("""
    SELECT 
        PULocationID,
        COUNT(*) as trip_count,
        ROUND(AVG(trip_distance), 2) as avg_distance,
        ROUND(AVG(total_amount), 2) as avg_fare
    FROM all_trips
    WHERE PULocationID IS NOT NULL
    GROUP BY PULocationID
    ORDER BY trip_count DESC
    LIMIT 10
""")

result.show()

In [ ]:
# Query 4: Payment type distribution
result = spark.sql("""
    SELECT 
        payment_type,
        COUNT(*) as trip_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
    FROM all_trips
    WHERE payment_type IS NOT NULL
    GROUP BY payment_type
    ORDER BY trip_count DESC
""")

result.show()

In [ ]:
# Query 5: Average revenue by hour of day
result = spark.sql("""
    SELECT 
        HOUR(pickup_datetime) as hour,
        COUNT(*) as trips,
        ROUND(AVG(total_amount), 2) as avg_revenue,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM all_trips
    GROUP BY HOUR(pickup_datetime)
    ORDER BY hour
""")

result.show(24)

In [ ]:
# Query 6: Long distance trips (> 20 miles)
result = spark.sql("""
    SELECT 
        service_type,
        pickup_datetime,
        PULocationID,
        DOLocationID,
        trip_distance,
        total_amount,
        ROUND(total_amount / trip_distance, 2) as cost_per_mile
    FROM all_trips
    WHERE trip_distance > 20
    ORDER BY trip_distance DESC
    LIMIT 20
""")

result.show(20, truncate=False)

In [ ]:
# Query 7: Weekend vs Weekday analysis
result = spark.sql("""
    SELECT 
        CASE 
            WHEN DAYOFWEEK(pickup_datetime) IN (1, 7) THEN 'Weekend'
            ELSE 'Weekday'
        END as day_type,
        service_type,
        COUNT(*) as trips,
        ROUND(AVG(trip_distance), 2) as avg_distance,
        ROUND(AVG(total_amount), 2) as avg_fare
    FROM all_trips
    GROUP BY 
        CASE 
            WHEN DAYOFWEEK(pickup_datetime) IN (1, 7) THEN 'Weekend'
            ELSE 'Weekday'
        END,
        service_type
    ORDER BY day_type, service_type
""")

result.show()

## DataFrames API vs SQL Comparison

The same query using both approaches:

In [ ]:
# Using SQL
print("Using SQL:")
sql_result = spark.sql("""
    SELECT 
        service_type,
        COUNT(*) as count
    FROM all_trips
    GROUP BY service_type
""")
sql_result.show()

# Using DataFrame API
print("\nUsing DataFrame API:")
df_result = df_all_trips \
    .groupBy('service_type') \
    .agg(F.count('*').alias('count'))
df_result.show()

## Save Results to Parquet

In [ ]:
# Save combined trips data to parquet (partitioned by service type)
# Uncomment to save
# df_all_trips.write \
#     .mode('overwrite') \
#     .partitionBy('service_type') \
#     .parquet('data/all_trips_2020')

# print("Saved to data/all_trips_2020/")

In [ ]:
# Stop Spark session when done
# spark.stop()